In [1]:
# import os
# from google.colab import drive

# # 1. Mount to the STANDARD location (safest option)
# drive.mount('/content/drive')

# # 2. Define your paths clearly
# project_folder = "diffusers_csc2210"
# # This is where the repo sits (inside your Drive)
# working_dir = f"/content/drive/MyDrive/{project_folder}"
# parent_dir = "/content/drive/MyDrive/"

# # 3. Move to Google Drive Root first
# %cd $parent_dir

# # 4. Smart Clone/Update Logic
# if not os.path.exists(project_folder):
#     print(f"Cloning into {project_folder}...")
#     !git clone https://github.com/yyyyy7105/diffusers_csc2210.git
# else:
#     print(f"Repo exists. Updating...")
#     # FIX: You must enter the specific folder BEFORE pulling
#     %cd $project_folder
#     !git pull
#     # Go back to parent to keep the flow consistent for the next step
#     %cd $parent_dir

# # 5. Enter the repo for installation
# %cd $working_dir

# # 6. Install in editable mode
# print("Installing in editable mode...")
# !pip install -e .

# !pip install git+https://github.com/yyyyy7105/diffusers_csc2210
# !pip install bitsandbytes transformers pillow

In [2]:
# import os
# from google.colab import drive
# import shutil # Import shutil for rmtree

# # Check if /content/drive exists and is a directory
# if os.path.isdir('/content/drive'):
#     # If it's a directory but NOT a mount point, it means it's a local folder
#     # that might contain residual files. We need to remove it to allow a clean mount.
#     if not os.path.ismount('/content/drive'):
#         print("Removing non-mountpoint /content/drive directory to allow clean mount...")
#         shutil.rmtree('/content/drive')

# # Now attempt to mount. Using force_remount=True can help in cases where Drive was previously mounted
# # but became unresponsive, forcing a fresh mount.
# drive.mount('/content/drive', force_remount=True)

# cache_dir = "/content/drive/MyDrive/huggingface_cache"
# os.makedirs(cache_dir, exist_ok=True)
# os.environ['HF_HOME'] = cache_dir
# os.environ['HF_HUB_DISABLE_SYMLINKS'] = '1'  # Crucial for Google Drive
# cache_dir = None

In [3]:
import torch
from diffusers import Flux2KleinPipeline, PipelineQuantizationConfig
from transformers import BitsAndBytesConfig

device = "cuda"
print(f"{torch.cuda.is_available()=}")
dtype = torch.float16
enable_profiler = True
print(f"{enable_profiler=}")

# Configure 4-bit quantization using BitsAndBytesConfig
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=dtype
)

cache_dir = r"D:\Users\14623\Documents\Coding\ml\flux2\flux_cache\models--black-forest-labs--FLUX.2-klein-4B\snapshots\5e67da950fce4a097bc150c22958a05716994cea"

# Wrap BitsAndBytesConfig in PipelineQuantizationConfig explicitly
quantization_config = PipelineQuantizationConfig(quant_backend="bitsandbytes_4bit", quant_kwargs=bnb_config.to_dict())

pipe = Flux2KleinPipeline.from_pretrained(cache_dir, quantization_config=quantization_config, torch_dtype=dtype, enable_profiler=enable_profiler)
pipe.enable_model_cpu_offload()

torch.cuda.is_available()=True
enable_profiler=True


Loading pipeline components...:   0%|          | 0/5 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

In [4]:
from PIL import Image
import torch


# Placeholder for your initial image. Replace with your actual image variable or path.
# For example: from PIL import Image; initial_image = Image.open("path/to/your/image.png")
# initial_image = Image.open("/content/Screenshot 2026-02-10 194248.png") # Please provide your initial image here. OPTIONAL

prompt = "better resolution, photo-realistic, a cat saying hello world"
image = pipe(
    prompt=prompt,
    # image=initial_image,  # OPTIONAL
    height=1024,
    width=1024,
    guidance_scale=1.0,
    num_inference_steps=4, # Since you are using 4 steps (likely Flux Schnell), keep this.
    generator=torch.Generator(device=device).manual_seed(0)
).images[0]
image.save("flux-klein.png")

I am running in: d:\Users\14623\Documents\gitRepo\diffusers_csc2210\2210
I am looking for logs at: d:\Users\14623\Documents\gitRepo\diffusers_csc2210\2210\log\flux2_profile


  0%|          | 0/4 [00:00<?, ?it/s]

-----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
                   Name    Self CPU %      Self CPU   CPU total %     CPU total  CPU time avg    # of Calls  
-----------------------  ------------  ------------  ------------  ------------  ------------  ------------  
         1_check_inputs         0.06%     446.800us         0.06%     446.800us     446.800us             1  
        3_encode_prompt        31.57%     217.161ms       177.00%        1.218s        1.218s             1  
       4_process_images         0.00%      28.200us         0.00%      28.200us      28.200us             1  
      5_prepare_latents         0.06%     380.400us         0.44%       3.020ms       3.020ms             1  
    6_prepare_timesteps         0.05%     318.500us         0.09%     628.000us     628.000us             1  
       7_denoising_loop         0.15%       1.025ms       656.98%        4.520s        4.520s             1  
     trans

In [5]:
profile = pipe.get_profile()